# HW 18: Link prediction v1

In [1]:
import networkx as nx
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
from multiprocessing import Pool
import time
import itertools
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score,
                             precision_recall_fscore_support,
                             precision_recall_curve,
                             average_precision_score,
                             precision_score)
from sklearn.preprocessing import StandardScaler

In [2]:
import matplotlib
font = {'family': 'sans-serif',
        'weight': 'bold',
        'size': 16}

matplotlib.rc('font', **font)

## Import data

In [3]:
df = pd.read_csv('data/social_network_11.csv')
df.head()

,i,j
0,2106,1
1,2132,1
2,3516,1
3,3881,1
4,10872,1


In [4]:
edges = df.apply(lambda x: (int(x[0]), int(x[1])), axis=1)
train_edges, test_edges = train_test_split(edges, test_size=0.2)

In [5]:
test_nodes = test_edges.apply(lambda x:x[0]).values.tolist() + \
    test_edges.apply(lambda x:x[1]).values.tolist()

In [6]:
len(train_edges), len(test_edges)

(27865, 6967)

In [7]:
e_df = pd.DataFrame(train_edges)

In [8]:
e_df['edge'] = 1

In [9]:
e_df.columns = ['nodes', 'edge']

## Creating a graph

In [10]:
G = nx.Graph()
G.add_edges_from(train_edges.values.tolist())
G.add_nodes_from(test_nodes)

In [11]:
print(nx.info(G))

Name: 
Type: Graph
Number of nodes: 6371
Number of edges: 25177
Average degree:   7.9036


Non-edges:

In [12]:
n_df = pd.Series(nx.non_edges(G))

In [13]:
all_df = pd.DataFrame(n_df)

In [14]:
all_df['edge'] = 0

In [15]:
all_df.columns = ['nodes', 'edge']

Sampling to make the final dataset more balanced:

In [16]:
all_df = all_df.sample(frac=0.03)

In [17]:
all_df.head()

,nodes,edge
4144515,"(718, 4006)",0
14759093,"(3286, 4867)",0
2059211,"(342, 8557)",0
11698567,"(2362, 4489)",0
16102245,"(3753, 5495)",0


In [18]:
all_df.shape

(607994, 2)

Explicitly defining test nodes

In [19]:
test_nodes = pd.DataFrame(test_edges)
test_nodes['edge'] = 0
test_nodes.columns = ['nodes', 'edge']
all_df = pd.concat([test_nodes, all_df], axis=0,
                   ignore_index=True)
all_df = all_df.drop_duplicates()

Creating a dataframe containing both edges and non-edges

In [20]:
all_df = pd.concat([e_df, all_df], axis=0,
                   ignore_index=True)

In [21]:
all_df.head()

,nodes,edge
0,"(3881, 8)",1
1,"(4860, 2862)",1
2,"(5298, 3887)",1
3,"(9258, 1958)",1
4,"(7881, 3449)",1


In [22]:
all_df.shape

(642585, 2)

In [23]:
all_df['edge'].value_counts()

0    614720
1     27865
Name: edge, dtype: int64

In [24]:
deg_df = pd.DataFrame.from_dict(dict(G.degree), orient='index')
deg_df = deg_df.reset_index()
deg_df.columns=['node','degree']
deg_df['clustering'] = nx.clustering(G, deg_df['node']).values()

In [25]:
deg_df.sort_values(by='degree', ascending=False)[:20]

,node,degree,clustering
73,62,422,0.011775
208,200,382,0.012038
75,1445,229,0.013867
99,106,207,0.026031
310,430,196,0.014547
301,58,166,0.038335
511,615,157,0.019680
186,683,132,0.020125
172,61,128,0.009966
235,118,126,0.018794


Reindexing with node number:

In [26]:
tmp = deg_df.set_index(deg_df['node'])

Adding node features to the graph dataframe:

In [27]:
col_list =  ['degree', 'clustering']

In [28]:
all_df.shape

(642585, 2)

In [29]:
all_df['node_1'] = all_df['nodes'].apply(lambda x: x[0])
all_df['node_2'] = all_df['nodes'].apply(lambda x: x[1])

In [30]:
for c in col_list:
    all_df.loc[:, c+'_1'] = tmp.loc[all_df['node_1'], c].values
    all_df.loc[:, c+'_2'] = tmp.loc[all_df['node_2'], c].values

In [31]:
all_df.head()

,nodes,edge,node_1,node_2,degree_1,degree_2,clustering_1,clustering_2
0,"(3881, 8)",1,3881,8,46,25,0.059903,0.143333
1,"(4860, 2862)",1,4860,2862,4,4,0.000000,0.000000
2,"(5298, 3887)",1,5298,3887,15,13,0.209524,0.089744
3,"(9258, 1958)",1,9258,1958,25,13,0.033333,0.051282
4,"(7881, 3449)",1,7881,3449,7,6,0.000000,0.066667


In [32]:
all_df.shape

(642585, 8)

Features characterizing node pairs:

In [33]:
all_df['res_all_index'] = pd.Series(
    nx.resource_allocation_index(G, all_df['nodes'].values))\
        .apply(lambda x:x[2])

In [34]:
all_df['jacc'] = pd.Series(
    nx.jaccard_coefficient(G, all_df['nodes'].values))\
        .apply(lambda x:x[2])

In [35]:
all_df['pref_att'] = pd.Series(
    nx.preferential_attachment(G, all_df['nodes'].values))\
        .apply(lambda x:x[2])

In [36]:
all_df = all_df.fillna(0)

In [37]:
all_df.head()

,nodes,edge,node_1,node_2,degree_1,degree_2,clustering_1,clustering_2,res_all_index,jacc,pref_att
0,"(3881, 8)",1,3881,8,46,25,0.059903,0.143333,0.244551,0.059701,1150
1,"(4860, 2862)",1,4860,2862,4,4,0.000000,0.000000,0.000000,0.000000,16
2,"(5298, 3887)",1,5298,3887,15,13,0.209524,0.089744,0.109480,0.120000,195
3,"(9258, 1958)",1,9258,1958,25,13,0.033333,0.051282,0.019231,0.027027,325
4,"(7881, 3449)",1,7881,3449,7,6,0.000000,0.066667,0.000000,0.000000,42


In [38]:
all_df = all_df.set_index(['nodes'])

In [39]:
test_df = all_df.loc[(test_edges), :]

In [40]:
test_df = test_df.loc[test_df['edge']==0, :]

In [41]:
train_df = all_df.loc[~all_df.index.isin(test_edges), :]

In [42]:
train_df.shape

(634444, 10)

In [43]:
test_df.shape

(6967, 10)

## Model

In [44]:
feature_col = train_df.columns[(train_df.columns != 'edge')]

In [45]:
target_col = ['edge']

In [46]:
X_train, X_test, y_train, y_test = train_df[feature_col], test_df[feature_col],\
train_df[target_col].values, np.ones(test_df[target_col].values.shape)

In [47]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [48]:
clf = LogisticRegression(solver='lbfgs', n_jobs=-1, class_weight='balanced')

In [49]:
clf.fit(X_train, y_train.ravel())

LogisticRegression(C=1.0, class_weight='balanced', dual=False,
                   fit_intercept=True, intercept_scaling=1, l1_ratio=None,
                   max_iter=100, multi_class='warn', n_jobs=-1, penalty='l2',
                   random_state=None, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)

In [50]:
clf.score(X_train, y_train)

0.9593849102521262

In [51]:
clf.score(X_test, y_test)

0.8778527343189321

In [52]:
sorted(list(zip(clf.coef_.tolist()[0], feature_col)), 
       key=lambda x: np.abs(x[0]), reverse=True)

[(-2.0615798185535867, 'node_2'),
 (1.9905484972642922, 'res_all_index'),
 (1.6701907442816126, 'node_1'),
 (1.2321689853893154, 'degree_1'),
 (0.36680640637939066, 'jacc'),
 (-0.31564942690748915, 'clustering_1'),
 (-0.22969680693863462, 'clustering_2'),
 (0.1997220794100904, 'degree_2'),
 (-0.19855995713965444, 'pref_att')]